In [ ]:
pip install -U llama-cpp-python

## Local Inference on GPU
Model page: https://huggingface.co/QuantFactory/Meta-Llama-3.1-8B-Instruct-GGUF

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/QuantFactory/Meta-Llama-3.1-8B-Instruct-GGUF)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# # !pip install llama-cpp-python

# from llama_cpp import Llama

# llm = Llama.from_pretrained(
# 	repo_id="QuantFactory/Meta-Llama-3.1-8B-Instruct-GGUF",
# 	filename="Meta-Llama-3.1-8B-Instruct.Q2_K.gguf",
# )


In [ ]:
# !pip install llama-cpp-python pandas

from llama_cpp import Llama
import pandas as pd
import json
import re
from tqdm import tqdm

# -------------------------------------------------------
# ✅ LOAD LLAMA WITH CUDA
# -------------------------------------------------------
llm = Llama.from_pretrained(
    repo_id="QuantFactory/Meta-Llama-3.1-8B-Instruct-GGUF",
    filename="Meta-Llama-3.1-8B-Instruct.Q2_K.gguf",

    n_gpu_layers=-1,   # Offload all layers to GPU
    n_threads=4,
    n_batch=256,

    use_mmap=False,
    use_mlock=True
)

# -------------------------------------------------------
# ✅ PROMPT TEMPLATE (STRICT JSON)
# -------------------------------------------------------
PROMPT = """
You are a clinical information extraction system.

Extract the drug names and adverse drug events from the text below.

Return ONLY valid JSON in this exact format:

{{
  "drug_names": ["drug1", "drug2"],
  "adverse_effects": ["effect1", "effect2"]
}}

Text:
{text}

ONLY RETURN THE JSON. NO EXTRA TEXT.
"""

# -------------------------------------------------------
# ✅ FUNCTION: ROBUST JSON EXTRACTION
# -------------------------------------------------------
def extract_drug_adr(text):
    prompt = PROMPT.format(text=text)

    result = llm(
        prompt=prompt,
        max_tokens=256,
        temperature=0.0
    )

    raw_output = result["choices"][0]["text"].strip()

    # Remove markdown fences
    cleaned = raw_output.replace("```json", "").replace("```", "").strip()

    # ✅ Extract *all* JSON objects, not just the first
    matches = re.findall(r"\{[\s\S]*?\}", cleaned)
    if matches:
        cleaned = matches[-1]   # ✅ pick the LAST JSON object

    # ✅ Try parsing JSON safely
    try:
        return json.loads(cleaned)
    except Exception:
        print("\n⚠️ JSON parse failed for text:", text)
        print("Model output:", raw_output)
        print("Cleaned JSON candidate:", cleaned)
        return {"drug_names": [], "adverse_effects": []}

# -------------------------------------------------------
# ✅ TEST ON A SAMPLE TEXT
# -------------------------------------------------------
text = "I am also allergic to all the penicillins!! I get rashes"
result = extract_drug_adr(text)

print("✅ Parsed JSON:", result)


In [ ]:
# !pip install llama-cpp-python pandas

from llama_cpp import Llama
import pandas as pd
import json
import re
from tqdm import tqdm

# -------------------------------------------------------
# ✅ LOAD LLAMA WITH CUDA
# -------------------------------------------------------
llm = Llama.from_pretrained(
    repo_id="QuantFactory/Meta-Llama-3.1-8B-Instruct-GGUF",
    filename="Meta-Llama-3.1-8B-Instruct.Q2_K.gguf",

    n_gpu_layers=-1,   # Offload all layers to GPU
    n_threads=4,
    n_batch=256,

    use_mmap=False,
    use_mlock=True
)

# -------------------------------------------------------
# ✅ PROMPT TEMPLATE (STRICT JSON)
# -------------------------------------------------------
PROMPT = """
You are a clinical information extraction system.

Extract the drug names and adverse drug events (ADEs) from the text below.

Return ONLY valid JSON in this exact format:

{{
  "drug_names": ["drug1", "drug2"],
  "adverse_effects": ["effect1", "effect2"]
}}

Text:
{text}

ONLY RETURN THE JSON. NO EXTRA TEXT.
"""

# -------------------------------------------------------
# ✅ ROBUST JSON EXTRACTION FUNCTION
# -------------------------------------------------------

def trim_text_for_context(text, max_chars=1200):
    """Trim text to fit within context size safely."""
    if len(text) <= max_chars:
        return text
    return text[:max_chars]


def extract_drug_adr(text):
    # ✅ trim long input
    trimmed = trim_text_for_context(text)

    prompt = PROMPT.format(text=trimmed)

    result = llm(
        prompt=prompt,
        max_tokens=256,
        temperature=0.0
    )

    raw_output = result["choices"][0]["text"].strip()

    cleaned = raw_output.replace("```json", "").replace("```", "").strip()

    matches = re.findall(r"\{[\s\S]*?\}", cleaned)
    if matches:
        cleaned = matches[-1]

    try:
        return json.loads(cleaned)
    except Exception:
        print("\n⚠️ JSON parse failed for text:", trimmed)
        print("Model output:", raw_output)
        print("Cleaned JSON candidate:", cleaned)
        return {"drug_names": [], "adverse_effects": []}



# -------------------------------------------------------
# ✅ PROCESS INPUT CSV
# -------------------------------------------------------
INPUT_CSV = "./data/LLaVA-Med/subset_of_all_ADR.csv"
OUTPUT_CSV = "./data/LLaVA-Med/output_all_adr_subset_0811_1.csv"

df = pd.read_csv(INPUT_CSV)

drug_col = []
adr_col = []

print("\n✅ Starting extraction over CSV...\n")

for text in tqdm(df["Preprocessed Posts"], desc="Extracting ADEs"):
    res = extract_drug_adr(str(text))
    drug_col.append(", ".join(res.get("drug_names", [])))
    adr_col.append(", ".join(res.get("adverse_effects", [])))

df["drug_names"] = drug_col
df["adverse_effects"] = adr_col

df.to_csv(OUTPUT_CSV, index=False)

print("\n✅ DONE! Output saved to:", OUTPUT_CSV)

In [ ]:
import pandas as pd

# ---------------------------
# Utility functions
# ---------------------------

def spans_overlap(span1, span2):
    """Return True if (start,end) tuples overlap by at least 1 character."""
    s1, e1 = span1
    s2, e2 = span2
    return not (e1 < s2 or e2 < s1)

def exact_match(pred, gold):
    return pred == gold

def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2*precision*recall / (precision + recall) if (precision+recall)>0 else 0.0
    return precision, recall, f1

# ---------------------------
# Core i2b2 Matching Logic
# ---------------------------

def score_entities(pred_list, gold_list, mode="strict"):
    """
    pred_list, gold_list = lists of (entity_string).
    For simplicity, we treat string spans as tokens:
    span = (start_index, end_index)
    """

    # Convert to spans: (start, end, surface_text)
    # This makes overlap and strict boundary comparisons possible.
    def to_spans(entities):
        spans = []
        start = 0
        for ent in entities:
            length = len(ent)
            spans.append(((start, start + length - 1), ent))
            start += length + 1   # space between items
        return spans

    pred_spans = to_spans(pred_list)
    gold_spans = to_spans(gold_list)

    matched_gold = set()
    TP = FP = FN = 0

    for (p_span, p_text) in pred_spans:
        found = False
        for i, (g_span, g_text) in enumerate(gold_spans):
            if i in matched_gold:
                continue

            # STRICT
            if mode == "strict":
                if p_text == g_text and p_span == g_span:
                    TP += 1
                    matched_gold.add(i)
                    found = True
                    break

            # RELAXED (overlap + same text)
            elif mode == "relaxed":
                if p_text == g_text and spans_overlap(p_span, g_span):
                    TP += 1
                    matched_gold.add(i)
                    found = True
                    break

        if not found:
            FP += 1

    # Remaining gold entities not matched
    FN = len(gold_spans) - len(matched_gold)

    return TP, FP, FN

# ---------------------------
# High-Level Evaluation
# ---------------------------

def evaluate_i2b2_style(df, pred_col, gold_col):
    TP_s = FP_s = FN_s = 0
    TP_r = FP_r = FN_r = 0

    for _, row in df.iterrows():
        pred = row[pred_col].split(",") if pd.notna(row[pred_col]) else []
        gold = row[gold_col].split(",") if pd.notna(row[gold_col]) else []

        # Clean whitespace
        pred = [x.strip() for x in pred if x.strip()]
        gold = [x.strip() for x in gold if x.strip()]

        # strict
        tp, fp, fn = score_entities(pred, gold, mode="strict")
        TP_s += tp; FP_s += fp; FN_s += fn

        # relaxed
        tp, fp, fn = score_entities(pred, gold, mode="relaxed")
        TP_r += tp; FP_r += fp; FN_r += fn

    strict = compute_metrics(TP_s, FP_s, FN_s)
    relaxed = compute_metrics(TP_r, FP_r, FN_r)

    return {
        "strict": {
            "precision": strict[0],
            "recall": strict[1],
            "f1": strict[2]
        },
        "relaxed": {
            "precision": relaxed[0],
            "recall": relaxed[1],
            "f1": relaxed[2]
        }
    }


In [ ]:
# Evaluate drug names
drug_results = evaluate_i2b2_style(df, "drug_pred", "drug_gold")

# Evaluate ADRs
adr_results = evaluate_i2b2_style(df, "adr_pred", "adr_gold")

print("Drug Results:", drug_results)
print("ADR Results:", adr_results)


In [ ]:
llm.create_chat_completion(
	messages = [
		{
			"role": "user",
			"content": "What is the capital of France?"
		}
	]
)